In [1]:
import pandas as pd
df = pd.read_csv("../data/transform/player_stats.csv")
p = pd.read_csv("../data/transform/players.csv")
m = pd.read_csv("../data/transform/matches.csv")

df = df.merge(
    p[['player_id', 'position']],
    on='player_id',
    how='left',
    suffixes=('', '_p')
)

df = df.merge(
    m[["match_id", "date", "home_club_id", "away_club_id", "home_goals", "away_goals"]],
    on='match_id',
    how='left',
    suffixes=('', '_m')
)



def get_result(row):
    if row["club_id"] == row["home_club_id"]:
        if row["home_goals"] > row["away_goals"]:
            return "win"
        elif row["home_goals"] < row["away_goals"]:
            return "loss"
        else:
            return "draw"
    
    elif row["club_id"] == row["away_club_id"]:
        if row["away_goals"] > row["home_goals"]:
            return "win"
        elif row["away_goals"] < row["home_goals"]:
            return "loss"
        else:
            return "draw"
    
    else:
        return None

df["result"] = df.apply(get_result, axis=1)


In [2]:
import pickle

with open("best_model.pkl", "rb") as f:
    model = pickle.load(f)

/opt/anaconda3/envs/tf/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.7.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/tf/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.7.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/tf/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.7.2 when using version 

In [3]:
features = [
    'goals',
    'assists',
    'yellow',
    'yellow_red',
    'red',
    'start_eleven',
    'minutes',
    'on_min',
    'off_min',
    'team_goals',
    'team_conceded',
    'position',
    'result'
]


def compute_rating(df, model, features):
    X = df[features]
    df['rating'] = model.predict(X)
    return df

compute_rating(df, model, features)


,player_id,match_id,club_id,goals,assists,yellow,yellow_red,red,start_eleven,minutes,...,team_goals,team_conceded,rating,position,date,home_club_id,away_club_id,home_goals,away_goals,result
0,284695,3393584,322,0,0,False,False,False,True,90,...,2,0,7.548952,Torwart,2020-08-15,322,8508,2,0,win
1,115188,3393584,322,0,0,False,False,False,True,90,...,2,0,7.427664,Innenverteidiger,2020-08-15,322,8508,2,0,win
2,19279,3393584,322,0,0,False,False,False,True,90,...,2,0,7.427664,Innenverteidiger,2020-08-15,322,8508,2,0,win
3,126514,3393584,322,0,0,False,False,False,True,90,...,2,0,7.275990,Linker Verteidiger,2020-08-15,322,8508,2,0,win
4,267582,3393584,322,0,0,False,False,False,True,90,...,2,0,7.146165,Rechter Verteidiger,2020-08-15,322,8508,2,0,win
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146959,1080572,4668667,14247,0,0,False,False,False,False,28,...,0,1,6.556970,Mittelfeld,2026-03-18,29727,14247,4,1,loss
146960,1193340,4668667,14247,0,0,False,False,False,False,6,...,0,1,6.498993,Innenverteidiger,2026-03-18,29727,14247,4,1,loss
146961,1091563,4668667,14247,0,0,False,False,False,False,10,...,0,1,6.530781,Defensives Mittelfeld,2026-03-18,29727,14247,4,1,loss
146962,357271,4668667,14247,0,0,False,False,False,False,6,...,0,1,6.598142,Zentrales Mittelfeld,2026-03-18,29727,14247,4,1,loss


In [4]:
df["rating"] = df["rating"].round(1)
cols = ["date",  "home_club_id", "away_club_id", "home_goals", "away_goals", "position_p", "result", "position"]
df = df.drop(columns = cols, errors="ignore")

In [5]:
df.to_csv("../data/transform/player_stats.csv", index=False)